In [1]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import Aer


# =========================
# HELPER
# =========================
def initialize_bits(qc, reg, bitstring):
    for i, b in enumerate(reversed(bitstring)):
        if b == '1':
            qc.x(reg[i])


# =========================
# ROUTER (FULL REVERSIBLE)
# =========================
def router(D_val, N1_val, N2_val):

    D  = QuantumRegister(3, 'D')
    N1 = QuantumRegister(3, 'N1')
    N2 = QuantumRegister(3, 'N2')

    X1 = QuantumRegister(3, 'X1')
    X2 = QuantumRegister(3, 'X2')

    d1 = QuantumRegister(2, 'd1')
    d2 = QuantumRegister(2, 'd2')

    anc = QuantumRegister(9, 'anc')
    flag = QuantumRegister(1, 'flag')

    OUT = QuantumRegister(3, 'OUT')
    c = ClassicalRegister(3, 'c')

    qc = QuantumCircuit(D, N1, N2, X1, X2, d1, d2, anc, flag, OUT, c)

    # INPUT
    initialize_bits(qc, D, D_val)
    initialize_bits(qc, N1, N1_val)
    initialize_bits(qc, N2, N2_val)

    # XOR
    for i in range(3):
        qc.cx(N1[i], X1[i])
        qc.cx(D[i],  X1[i])

        qc.cx(N2[i], X2[i])
        qc.cx(D[i],  X2[i])

    # POPCOUNT d1
    qc.cx(X1[0], d1[0])
    qc.cx(X1[1], d1[0])
    qc.cx(X1[2], d1[0])

    qc.ccx(X1[0], X1[1], anc[0])
    qc.ccx(X1[0], X1[2], anc[1])
    qc.ccx(X1[1], X1[2], anc[2])

    qc.cx(anc[0], d1[1])
    qc.cx(anc[1], d1[1])
    qc.cx(anc[2], d1[1])

    # POPCOUNT d2
    qc.cx(X2[0], d2[0])
    qc.cx(X2[1], d2[0])
    qc.cx(X2[2], d2[0])

    qc.ccx(X2[0], X2[1], anc[3])
    qc.ccx(X2[0], X2[2], anc[4])
    qc.ccx(X2[1], X2[2], anc[5])

    qc.cx(anc[3], d2[1])
    qc.cx(anc[4], d2[1])
    qc.cx(anc[5], d2[1])

    # =========================
    # COMPARATOR (d1 > d2)
    # =========================
    qc.x(d2[1])
    qc.ccx(d1[1], d2[1], flag[0])
    qc.x(d2[1])

    qc.cx(d1[1], anc[6])
    qc.cx(d2[1], anc[6])
    qc.x(anc[6])

    qc.x(d2[0])
    qc.ccx(d1[0], d2[0], anc[7])
    qc.x(d2[0])

    qc.ccx(anc[6], anc[7], anc[8])
    qc.cx(anc[8], flag[0])

    # =========================
    # SELECTION
    # =========================
    for i in range(3):
        qc.ccx(flag[0], N2[i], OUT[i])
        qc.x(flag[0])
        qc.ccx(flag[0], N1[i], OUT[i])
        qc.x(flag[0])

    # =========================
    # UNCOMPUTE
    # =========================

    qc.cx(anc[8], flag[0])
    qc.ccx(anc[6], anc[7], anc[8])

    qc.x(d2[0])
    qc.ccx(d1[0], d2[0], anc[7])
    qc.x(d2[0])

    qc.x(anc[6])
    qc.cx(d2[1], anc[6])
    qc.cx(d1[1], anc[6])

    qc.x(d2[1])
    qc.ccx(d1[1], d2[1], flag[0])
    qc.x(d2[1])

    qc.cx(anc[5], d2[1])
    qc.cx(anc[4], d2[1])
    qc.cx(anc[3], d2[1])

    qc.ccx(X2[1], X2[2], anc[5])
    qc.ccx(X2[0], X2[2], anc[4])
    qc.ccx(X2[0], X2[1], anc[3])

    qc.cx(X2[2], d2[0])
    qc.cx(X2[1], d2[0])
    qc.cx(X2[0], d2[0])

    qc.cx(anc[2], d1[1])
    qc.cx(anc[1], d1[1])
    qc.cx(anc[0], d1[1])

    qc.ccx(X1[1], X1[2], anc[2])
    qc.ccx(X1[0], X1[2], anc[1])
    qc.ccx(X1[0], X1[1], anc[0])

    qc.cx(X1[2], d1[0])
    qc.cx(X1[1], d1[0])
    qc.cx(X1[0], d1[0])

    for i in range(3):
        qc.cx(D[i],  X2[i])
        qc.cx(N2[i], X2[i])

        qc.cx(D[i],  X1[i])
        qc.cx(N1[i], X1[i])

    # MEASURE
    for i in range(3):
        qc.measure(OUT[i], c[i])

    return qc


# =========================
# TEST
# =========================
def hamming(a, b):
    return sum(x != y for x, y in zip(a, b))


def run_test(D, N1, N2):

    qc = router(D, N1, N2)

    backend = Aer.get_backend('aer_simulator')
    backend.set_options(method="matrix_product_state")

    result = backend.run(qc, shots=1).result()

    raw = list(result.get_counts().keys())[0]

    # ✅ FIX: do NOT reverse
    measured = raw

    d1 = hamming(D, N1)
    d2 = hamming(D, N2)

    expected = N2 if d1 > d2 else N1

    print("====================================")
    print(f"D={D}  N1={N1}  N2={N2}")
    print(f"d1={d1}, d2={d2}")
    print(f"Expected: {expected}")
    print(f"Measured: {measured}")

    if measured == expected:
        print("✔ PASS")
    else:
        print("❌ FAIL")


# =========================
# BASIC TEST CASES
# =========================
def run_basic_tests():
    tests = [
        ("000", "001", "111"),
        ("101", "111", "001"),
        ("111", "000", "110"),
        ("010", "011", "110"),
        ("001", "010", "011"),
        ("110", "100", "111"),
    ]

    print("\nRunning BASIC tests...\n")

    for t in tests:
        run_test(*t)


# RUN
run_basic_tests()


Running BASIC tests...

D=000  N1=001  N2=111
d1=1, d2=3
Expected: 001
Measured: 001
✔ PASS
D=101  N1=111  N2=001
d1=1, d2=1
Expected: 111
Measured: 111
✔ PASS
D=111  N1=000  N2=110
d1=3, d2=1
Expected: 110
Measured: 110
✔ PASS
D=010  N1=011  N2=110
d1=1, d2=1
Expected: 011
Measured: 011
✔ PASS
D=001  N1=010  N2=011
d1=2, d2=1
Expected: 011
Measured: 011
✔ PASS
D=110  N1=100  N2=111
d1=1, d2=1
Expected: 100
Measured: 100
✔ PASS
